# 04 · E no SQL Server 2025? E no Azure SQL?

Nos notebooks 01 a 03 fizemos busca semântica no **SQL Server 2022**, que não tem tipo de vetor. Este notebook responde à pergunta que todo aluno faz no final:

> *"E se eu tivesse o SQL Server 2025, que tem o tipo `VECTOR`? Fica tudo resolvido?"*

Resposta curta: **a busca exata fica ótima; o índice vetorial ainda tem uma pegadinha no SQL Server instalado na sua máquina (on-premise) — e ela já foi resolvida no Azure SQL.**

| Símbolo | O que significa |
|---|---|
| 🟢 **ao vivo** | roda agora, no nosso SQL Server 2022 |
| 🔵 **roteiro 2025 on-premise** | T-SQL para ler e discutir — **não roda** no 2022 |
| ☁️ **roteiro Azure SQL** | T-SQL do Azure SQL Database — **não roda** no 2022 |

> Pré-requisito: notebooks 01 e 02 executados (banco `RagPoliticos` com as frases e os embeddings).

In [1]:
import sys, pathlib, warnings, time, json
warnings.filterwarnings("ignore", category=UserWarning)   # aviso do pandas sobre conexão pyodbc
sys.path.append(str(pathlib.Path.cwd().parent))   # permite "from src import ..."
from src import config, banco
from src.vetores import IndiceEmMemoria, bytes_para_vetor, vetor_para_bytes
import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

conn = banco.conectar(config.SQL_DATABASE)
modelo = SentenceTransformer(config.MODELO_EMBEDDING)
print(conn.execute("SELECT @@VERSION").fetchval().splitlines()[0])

c:\dev\Estudo-MDS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2375.30it/s]


Microsoft SQL Server 2022 (RTM-GDR) (KB5122771) - 16.0.1200.5 (X64) 


## 1. Recapitulando: onde cada coisa acontece hoje

| Etapa | Nosso SQL Server 2022 |
|---|---|
| Transformar texto em vetor | a **aplicação** (modelo Python) |
| Guardar o vetor | o **banco**, em `VARBINARY` (bytes que ele não entende) |
| Calcular a distância | a **aplicação** (NumPy), ou o banco "na mão" com `GENERATE_SERIES` (notebook 01) |

Nos dois casos é **força bruta**: a pergunta é comparada com **todas** as frases.

## 2. 🟢 Ao vivo: o 2022 não conhece `VECTOR` nem `VECTOR_DISTANCE`

As duas linhas abaixo são T-SQL válido no **SQL Server 2025** e no **Azure SQL**. No 2022, falham.

In [2]:
testes = {
    "tipo VECTOR":     "DECLARE @v VECTOR(3) = '[0.1, 0.2, 0.3]'; SELECT @v;",
    "VECTOR_DISTANCE": "SELECT VECTOR_DISTANCE('cosine', CAST('[1, 0]' AS VECTOR(2)), CAST('[0, 1]' AS VECTOR(2)));",
}
for nome, sql in testes.items():
    try:
        conn.execute(sql).fetchall()
        print(f"{nome}: funcionou (este servidor já é 2025 ou Azure SQL)")
    except Exception as e:
        msg = str(e).split("[SQL Server]")[-1].split("(")[0].strip()
        print(f"{nome}: ❌ {msg}")

tipo VECTOR: ❌ Parameter or variable '@v' has an invalid data type.
VECTOR_DISTANCE: ❌ 'VECTOR_DISTANCE' is not a recognized built-in function name.


## 3. 🟢 Ao vivo: a força bruta nos dois lugares possíveis do 2022

Mesma pergunta, mesma conta, dois lugares. Repare também **quanto dado sai do banco** em cada caso.

In [3]:
pergunta = "coragem para enfrentar o pânico"
vq = modelo.encode(pergunta, normalize_embeddings=True)

# A) aplicação: carrega TODOS os vetores uma vez e compara em memória
t = time.perf_counter()
indice = IndiceEmMemoria.do_banco(conn, config.MODELO_EMBEDDING)
carga_ms = (time.perf_counter() - t) * 1000
t = time.perf_counter()
for _ in range(100):
    indice.buscar(vq, 5)
python_ms = (time.perf_counter() - t) * 1000 / 100

# B) banco: a procedure do notebook 01 abre os bytes com GENERATE_SERIES
t = time.perf_counter()
for _ in range(5):
    tsql = pd.read_sql("EXEC rag.usp_BuscaSemanticaTSQL @Vetor = ?, @Modelo = ?, @K = 5",
                       conn, params=[vetor_para_bytes(vq), config.MODELO_EMBEDDING])
tsql_ms = (time.perf_counter() - t) * 1000 / 5

pd.DataFrame([
    {"Onde": "A) Aplicação (NumPy)", "Tempo por busca (ms)": round(python_ms, 3),
     "O que sai do banco": f"todos os vetores, uma vez ({indice.matriz.nbytes/1024:.0f} KB; carga de {carga_ms:.0f} ms)"},
    {"Onde": "B) Banco (T-SQL + GENERATE_SERIES)", "Tempo por busca (ms)": round(tsql_ms, 1),
     "O que sai do banco": "só as 5 respostas"},
])

,Onde,Tempo por busca (ms),O que sai do banco
0,A) Aplicação (NumPy),0.032,"todos os vetores, uma vez (46 KB; carga de 9 ms)"
1,B) Banco (T-SQL + GENERATE_SERIES),420.300,só as 5 respostas


**Leitura:** no 2022, a opção rápida obriga a levar todos os vetores para a aplicação; a opção que não leva nada é lenta, porque o banco faz "na mão" uma conta para a qual não foi feito.

É exatamente esse dilema que o SQL Server 2025 resolve — até certo ponto.

## 4. 🔵 Roteiro: a mesma aula no SQL Server 2025 on-premise

> Os blocos desta seção são **roteiro**: leia, discuta e copie para um SQL Server 2025. Não rodam no nosso 2022.

### 4.1 A tabela com o tipo nativo `VECTOR`

```sql
CREATE TABLE rag.FraseVetor (
    FraseId INT NOT NULL CONSTRAINT PK_FraseVetor PRIMARY KEY CLUSTERED  -- (1)
            CONSTRAINT FK_FraseVetor_Frase REFERENCES rag.Frase(FraseId),
    Vetor   VECTOR(384) NOT NULL                                          -- (2)
);
```

1. **Chave primária clusterizada numa única coluna `INT`.** O índice vetorial exige isso. A nossa `rag.FraseEmbedding`, com chave `(FraseId, Modelo)`, **não serviria**:
   *Msg 42217 — "Table 'FraseEmbedding' must have a clustered primary key on a single 4 byte INT column to create a vector index."*
2. `VECTOR(384)` guarda os mesmos float32 de 4 bytes do nosso `VARBINARY` — mas agora **o banco sabe o que são** e valida as 384 dimensões sozinho (adeus `CHECK DATALENGTH(...) = Dimensoes * 4`).

Pela documentação, o tipo `VECTOR` é criado e exibido como um **array JSON** (`'[0.12, -0.5, ...]'`) e converte de/para `varchar`, `nvarchar` e `json`. O caminho natural de migração passa pela aplicação: ela lê os nossos bytes e manda o JSON. 🟢 Veja como fica um vetor nosso nesse formato:

In [ ]:
v1 = conn.execute("SELECT Vetor FROM rag.FraseEmbedding WHERE FraseId = 1 AND Modelo = ?",
                  config.MODELO_EMBEDDING).fetchval()
como_json = json.dumps([round(float(x), 6) for x in bytes_para_vetor(v1)])
print(f"{len(bytes_para_vetor(v1))} números · início do JSON que iria para VECTOR(384):")
print(como_json[:110] + " ...")

### 4.2 A busca exata com `VECTOR_DISTANCE` — funciona muito bem

```sql
DECLARE @pergunta VECTOR(384) = @vetor_json_da_aplicacao;   -- ou AI_GENERATE_EMBEDDINGS (seção 5)

SELECT TOP (5) f.FraseId, f.Texto,
       VECTOR_DISTANCE('cosine', @pergunta, fv.Vetor) AS distancia   -- 0 = idêntico
FROM rag.FraseVetor fv
JOIN rag.Frase f ON f.FraseId = fv.FraseId
ORDER BY distancia;
```

- Continua sendo **força bruta** (compara com todas as linhas) — mas **dentro do banco**, em código nativo.
- Nada é carregado na aplicação; frase nova aparece na hora; dá para misturar com `WHERE` comum.
- A Microsoft recomenda a busca exata para até **~50 mil vetores**. **Para o tamanho da nossa aula, é a escolha certa.**

### 4.3 Criar o índice vetorial (DiskANN) — ainda em *preview*

```sql
ALTER DATABASE SCOPED CONFIGURATION SET PREVIEW_FEATURES = ON;   -- obrigatório no SQL Server 2025

CREATE VECTOR INDEX ix_FraseVetor ON rag.FraseVetor (Vetor)
    WITH (METRIC = 'cosine', TYPE = 'DiskANN');
```

Com as nossas 22 frases, isso **falha**: o índice exige pelo menos **100 linhas**.

```text
Msg 42266, Level 16, State 1
Cannot create a vector index. The table contains only 22 rows with non-null vectors,
but at least 100 are required for vector index creation.
```

### 4.4 Supondo 100+ frases: a busca aproximada

No SQL Server 2025 o índice é da **versão antiga**, que usa o parâmetro `TOP_N`:

```sql
SELECT TOP (5) fv.FraseId, s.distance
FROM VECTOR_SEARCH(
        TABLE      = rag.FraseVetor AS fv,
        COLUMN     = Vetor,
        SIMILAR_TO = @pergunta,
        METRIC     = 'cosine',
        TOP_N      = 5
     ) AS s
ORDER BY s.distance;
```

Rápido, sem varrer a tabela inteira — em troca, **aproximado** (pode perder um vizinho de verdade).

### 4.5 ⚠️ A pegadinha: a tabela vira somente leitura

Com o índice criado, tente cadastrar uma frase nova — exatamente o que o nosso app faz em *"Cadastrar uma nova frase"*:

```sql
INSERT INTO rag.FraseVetor (FraseId, Vetor) VALUES (@id, @vetor);
```

```text
Msg 42231, Level 16
Data modification statement failed because table 'FraseVetor' has a vector index on it.
```

**Nenhum** `INSERT`, `UPDATE`, `DELETE` ou `MERGE` passa. `TRUNCATE TABLE` também não.

### 4.6 E como atualizo os dados, então?

```sql
DROP INDEX ix_FraseVetor ON rag.FraseVetor;          -- 1. apaga o índice (a busca aproximada para!)
INSERT INTO rag.FraseVetor (FraseId, Vetor) VALUES (@id, @vetor);   -- 2. altera os dados
CREATE VECTOR INDEX ix_FraseVetor ON rag.FraseVetor (Vetor)
    WITH (METRIC = 'cosine', TYPE = 'DiskANN');      -- 3. reconstrói TUDO do zero
```

E a "válvula de escape" do Azure, `ALLOW_STALE_VECTOR_INDEX = ON` (gravar aceitando um índice desatualizado), **não existe no SQL Server 2025**.

**Consequência para o nosso app:** num SQL Server 2025 on-premise com índice vetorial, a demonstração *"Cadastrar uma nova frase ao vivo"* **quebraria**. Com `VECTOR_DISTANCE` (sem índice), funciona normalmente.

## 5. ☁️ Roteiro: no Azure SQL Database a pegadinha já foi resolvida

O Azure SQL Database (e o SQL database do Microsoft Fabric) já recebeu a **versão 3** do índice vetorial. A Microsoft deixa isso explícito: *"The latest version of Vector Indexes is only available in Azure SQL Database and SQL database in Microsoft Fabric currently."*

### 5.1 O mesmo `CREATE`, outro resultado

```sql
-- sem PREVIEW_FEATURES: no Azure não é preciso ligar
CREATE VECTOR INDEX ix_FraseVetor ON rag.FraseVetor (Vetor)
    WITH (METRIC = 'cosine', TYPE = 'DiskANN');

-- Qual versão foi criada? (3 = a nova)
SELECT i.name, JSON_VALUE(v.build_parameters, '$.Version') AS versao_indice
FROM sys.vector_indexes v
JOIN sys.indexes i ON i.object_id = v.object_id AND i.index_id = v.index_id;
```

### 5.2 Agora o `INSERT` funciona — e a frase já aparece na busca

```sql
INSERT INTO rag.FraseVetor (FraseId, Vetor) VALUES (@id, @vetor);   -- ✅ sem erro
-- o índice se atualiza sozinho: a frase entra na próxima busca, sem reconstruir nada
```

### 5.3 A consulta muda um pouco: `WITH APPROXIMATE` no lugar de `TOP_N`

```sql
SELECT TOP (5) WITH APPROXIMATE fv.FraseId, f.Texto, s.distance
FROM VECTOR_SEARCH(
        TABLE      = rag.FraseVetor AS fv,
        COLUMN     = Vetor,
        SIMILAR_TO = @pergunta,
        METRIC     = 'cosine'
     ) AS s
JOIN rag.Frase f ON f.FraseId = fv.FraseId
WHERE f.Ano >= 1950            -- filtro comum na mesma consulta
ORDER BY s.distance;
```

Outra melhoria da versão 3: filtros sobre colunas da **própria tabela vetorial** são aplicados **durante** a busca (*iterative filtering*). Na versão antiga (a do SQL Server 2025), o filtro só vinha **depois** — e podia devolver menos linhas do que o `TOP` pedido.

Usar a sintaxe antiga (`TOP_N`) num índice versão 3 dá erro:

```text
Msg 42274, Level 16, State 1
Vector search with version 3 index does not support explicit TOP_N parameter.
```

### 5.4 Bônus: o banco pedindo o vetor ao modelo

Tanto no 2025 quanto no Azure, o próprio banco pode chamar um modelo de embeddings cadastrado com `CREATE EXTERNAL MODEL`:

```sql
DECLARE @pergunta VECTOR(384) = AI_GENERATE_EMBEDDINGS(N'coragem para enfrentar o pânico' USE MODEL MeuModelo);
```

O modelo continua sendo um serviço **fora** do banco (Azure OpenAI, Ollama…) — o SQL Server só aprendeu a chamá-lo. Atenção: o `VECTOR(n)` precisa ter as dimensões **do modelo usado**.

## 6. O quadro para levar para casa

| | Onde a distância é calculada | Força bruta? | Cadastrar frase ao vivo | Para quantos vetores |
|---|---|---|---|---|
| **2022 — app (nossa aula)** | aplicação (NumPy) | sim | ✅ (depois de recarregar o cache) | até centenas de milhares (cabe na RAM) |
| **2022 — T-SQL caseiro** | banco, "na mão" | sim | ✅ na hora | pequeno: é lento |
| **2025 — `VECTOR_DISTANCE`** | banco, nativo | sim | ✅ na hora | até ~50 mil (recomendação da Microsoft) |
| **2025 on-premise — índice** (*preview*) | banco, aproximado | não | ❌ **tabela somente leitura** | grandes volumes, dados estáticos |
| **Azure SQL — índice v3** | banco, aproximado | não | ✅ índice se atualiza sozinho | grandes volumes, dados vivos |

> **No 2022, o SQL Server guarda e a aplicação pensa. No 2025, o SQL Server guarda e pensa — mas o índice vetorial on-premise ainda não aceita dados mudando. No Azure, já aceita.**

## 7. Para discutir com a turma

1. Nosso app cadastra frases ao vivo. Num SQL Server 2025 **on-premise**, você usaria índice vetorial? Por quê?
2. Um catálogo de 5 milhões de produtos que muda uma vez por noite. Índice on-premise resolveria? Como ficaria a carga noturna? *(dica: `DROP INDEX` → carga → `CREATE VECTOR INDEX`)*
3. Por que a Microsoft exige 100 linhas para criar o índice e ainda recomenda busca exata para até ~50 mil vetores?
4. Recursos em *preview* em produção: quais riscos você correria?

### Fontes

- Microsoft Learn — [CREATE VECTOR INDEX](https://learn.microsoft.com/sql/t-sql/statements/create-vector-index-transact-sql?view=sql-server-ver17) (versão 3 só no Azure SQL/Fabric; tabelas somente leitura na versão anterior; mínimo de 100 linhas)
- Microsoft Learn — [VECTOR_SEARCH](https://learn.microsoft.com/sql/t-sql/functions/vector-search-transact-sql?view=sql-server-ver17) e [VECTOR_DISTANCE](https://learn.microsoft.com/sql/t-sql/functions/vector-distance-transact-sql?view=sql-server-ver17)
- Microsoft Learn — [ALTER DATABASE SCOPED CONFIGURATION: ALLOW_STALE_VECTOR_INDEX](https://learn.microsoft.com/sql/t-sql/statements/alter-database-scoped-configuration-transact-sql?view=sql-server-ver17) (indisponível no SQL Server 2025)
- Microsoft Learn — [Erros 41400 a 49999](https://learn.microsoft.com/sql/relational-databases/errors-events/database-engine-events-and-errors-41400-to-49999?view=sql-server-ver17) (42217, 42231)
- Microsoft Learn — [Vector search and vector indexes](https://learn.microsoft.com/sql/sql-server/ai/vectors?view=sql-server-ver17) (busca exata para até ~50 mil vetores)
- Redgate Simple Talk — [Vector search in SQL Server: VECTOR_DISTANCE, VECTOR_SEARCH, and index trade-offs](https://www.red-gate.com/simple-talk/databases/sql-server/vector-search-in-sql-server-vector_distance-vector_search-and-index-trade-offs/)

*Recursos em preview mudam rápido: confira a documentação antes de cada turma.*